# Data Loading

Este notebook tiene como objetivo verificar el entorno de trabajo,
descargar o localizar los datos originales de la competencia y realizar
una primera inspección de los archivos disponibles.

En esta etapa no se realizan transformaciones ni tareas de limpieza.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib
import sklearn
import kagglehub


In [2]:
print("Entorno cargado correctamente.\n")

print("Versiones utilizadas:")
print(f"pandas:       {pd.__version__}")
print(f"numpy:        {np.__version__}")
print(f"matplotlib:   {matplotlib.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

Entorno cargado correctamente.

Versiones utilizadas:
pandas:       2.3.3
numpy:        2.2.6
matplotlib:   3.10.9
scikit-learn: 1.7.2


In [3]:
PROJECT_ROOT = Path("..").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
INTERIM_DATA_DIR = DATA_DIR / "interim"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Ruta raíz del proyecto:")
print(PROJECT_ROOT)

print("\nRuta de datos originales:")
print(RAW_DATA_DIR)

Ruta raíz del proyecto:
C:\Projects\proyecto2_DS

Ruta de datos originales:
C:\Projects\proyecto2_DS\data\raw


In [4]:
EXPECTED_FILES = [
    "summaries_train.csv",
    "summaries_test.csv",
    "prompts_train.csv",
    "prompts_test.csv",
    "sample_submission.csv",
]

print("Archivos esperados:")

for filename in EXPECTED_FILES:
    print(f"- {filename}")

Archivos esperados:
- summaries_train.csv
- summaries_test.csv
- prompts_train.csv
- prompts_test.csv
- sample_submission.csv


In [7]:
files_status = {
    filename: (RAW_DATA_DIR / filename).exists()
    for filename in EXPECTED_FILES
}

for filename, exists in files_status.items():
    status = "OK" if exists else "FALTA"
    print(f"{status:5} | {filename}")

all_files_exist = all(files_status.values())

print(f"\nDataset completo disponible: {all_files_exist}")

OK    | summaries_train.csv
OK    | summaries_test.csv
OK    | prompts_train.csv
OK    | prompts_test.csv
OK    | sample_submission.csv

Dataset completo disponible: True


In [8]:
if not all_files_exist:
    print("Descargando CommonLit - Evaluate Student Summaries...\n")

    download_path = kagglehub.competition_download(
        "commonlit-evaluate-student-summaries",
        output_dir=str(RAW_DATA_DIR),
    )

    print("\nDescarga finalizada.")
    print(f"Archivos almacenados en: {download_path}")

else:
    print("Los archivos originales ya existen en data/raw/.")
    print("No es necesario descargarlos nuevamente.")

Los archivos originales ya existen en data/raw/.
No es necesario descargarlos nuevamente.


In [9]:
missing_files = [
    filename
    for filename in EXPECTED_FILES
    if not (RAW_DATA_DIR / filename).exists()
]

if missing_files:
    raise FileNotFoundError(
        "No se encontraron los siguientes archivos:\n"
        + "\n".join(missing_files)
    )

print("Validación completada.")
print("Todos los archivos esperados se encuentran en data/raw/.")

Validación completada.
Todos los archivos esperados se encuentran en data/raw/.


In [10]:
print("Contenido de data/raw/:\n")

for file_path in sorted(RAW_DATA_DIR.iterdir()):
    if file_path.is_file():
        size_mb = file_path.stat().st_size / (1024 ** 2)
        print(f"{file_path.name:30} {size_mb:.2f} MB")

Contenido de data/raw/:

prompts_test.csv               0.00 MB
prompts_train.csv              0.02 MB
README.md                      0.00 MB
sample_submission.csv          0.00 MB
summaries_test.csv             0.00 MB
summaries_train.csv            3.27 MB


In [11]:
summaries_train = pd.read_csv(
    RAW_DATA_DIR / "summaries_train.csv"
)

summaries_test = pd.read_csv(
    RAW_DATA_DIR / "summaries_test.csv"
)

prompts_train = pd.read_csv(
    RAW_DATA_DIR / "prompts_train.csv"
)

prompts_test = pd.read_csv(
    RAW_DATA_DIR / "prompts_test.csv"
)

sample_submission = pd.read_csv(
    RAW_DATA_DIR / "sample_submission.csv"
)

print("Todos los archivos CSV fueron cargados correctamente.")

Todos los archivos CSV fueron cargados correctamente.


In [15]:
datasets = {
    "summaries_train": summaries_train,
    "summaries_test": summaries_test,
    "prompts_train": prompts_train,
    "prompts_test": prompts_test,
    "sample_submission": sample_submission,
}

In [16]:
print("Dimensiones de los datasets:\n")

for name, df in datasets.items():
    rows, columns = df.shape

    print(
        f"{name:20} "
        f"{rows:>6} filas x "
        f"{columns:>2} columnas"
    )

Dimensiones de los datasets:

summaries_train        7165 filas x  5 columnas
summaries_test            4 filas x  3 columnas
prompts_train             4 filas x  4 columnas
prompts_test              2 filas x  4 columnas
sample_submission         4 filas x  3 columnas


In [17]:
for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * len(name))

    for column in df.columns:
        print(f"- {column}")


summaries_train
---------------
- student_id
- prompt_id
- text
- content
- wording

summaries_test
--------------
- student_id
- prompt_id
- text

prompts_train
-------------
- prompt_id
- prompt_question
- prompt_title
- prompt_text

prompts_test
------------
- prompt_id
- prompt_question
- prompt_title
- prompt_text

sample_submission
-----------------
- student_id
- content
- wording


In [22]:
summaries_train.head()


,student_id,prompt_id,text,content,wording
0,000e8c3c7ddb,814d6b,The third wave was an experimentto see how peo...,0.205683,0.380538
1,0020ae56ffbf,ebad26,They would rub it up with soda to make the sme...,-0.548304,0.506755
2,004e978e639e,3b9047,"In Egypt, there were many occupations and soci...",3.128928,4.231226
3,005ab0199905,3b9047,The highest class was Pharaohs these people we...,-0.210614,-0.471415
4,0070c9e7af47,814d6b,The Third Wave developed rapidly because the ...,3.272894,3.219757


In [20]:
prompts_train.head()


,prompt_id,prompt_question,prompt_title,prompt_text
0,39c16e,Summarize at least 3 elements of an ideal trag...,On Tragedy,Chapter 13 \r\nAs the sequel to what has alrea...
1,3b9047,"In complete sentences, summarize the structure...",Egyptian Social Structure,Egyptian society was structured like a pyramid...
2,814d6b,Summarize how the Third Wave developed over su...,The Third Wave,Background \r\nThe Third Wave experiment took ...
3,ebad26,Summarize the various ways the factory would u...,Excerpt from The Jungle,"With one member trimming beef in a cannery, an..."


In [21]:
summaries_test.head()


,student_id,prompt_id,text
0,000000ffffff,abc123,Example text 1
1,111111eeeeee,def789,Example text 2
2,222222cccccc,abc123,Example text 3
3,333333dddddd,def789,Example text 4


In [23]:
prompts_test.head()

,prompt_id,prompt_question,prompt_title,prompt_text
0,abc123,Summarize...,Example Title 1,Heading\nText...
1,def789,Summarize...,Example Title 2,Heading\nText...
